# activation patching
**Question:** if we copy the model's internal state for "**icy**" into a story that says "the road was **long**" (no other words change), does the model then reason like *seeing* the icy road (more snow, bus late) or like *doing* it (bus late, snow unchanged)?

This is the method causal-tracing work uses, so it tests that work's assumption directly.

**Stories:** each mediator sentence comes in three versions that differ by one word: *true* ("icy"), *false* ("dry") and *neutral* ("long", an irrelevant property). The base story uses the neutral word.

**Patch:** at one layer, replace the base run's internal state with the state from a source run:
* **word**: the last token of the key word; **sentence**: from that token to the end of the sentence.
* Sources: **true** (same story with "icy"), **false** (same story with "dry"), **unrelated** (another scenario's true story, same variant).

**Controls:** three filler questions (yes-bias, subtracted); the unrelated source; the false source (should push the mediator answer *down*).

**Pass rule, fixed before running.** A (layer, mode) setting passes if, averaged over scenarios: the true patch moves the filler-corrected mediator answer by ≥ 1.0; it beats the unrelated patch by > 0.5 on that answer, in ≥ 70% of scenarios; the filler moves < 0.5; and the false patch moves the mediator answer below 0.
**Story filters (same as before, filler-corrected, per scenario):** Δcause(see) > 0.5, Δeffect(see) > 0.5, Δeffect(do) > 0.5, r_see − r_do > 0.25.

Set **SET** in the settings cell: `"exploration"` first (run 1's 20 scenarios), then `"confirmation"` (25 new scenarios). Results go to `results/P_<SET>_*`. **Runtime → Run all.** T4, ~25 min for exploration, ~45 min for confirmation.

In [ ]:
# Mount Google Drive and work inside the see-vs-do folder (story files live there; results are saved there)
import os
from google.colab import drive
drive.mount("/content/drive")
os.chdir("/content/drive/MyDrive/see-vs-do")
OUT = "results"
os.makedirs(OUT, exist_ok=True)
print("working in", os.getcwd(), "| files:", sorted(os.listdir(".")))

In [ ]:
# ---------- Setup: model, tokenizer, settings
import gc, os, time, torch, numpy as np, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.set_grad_enabled(False)

NAME = "Qwen/Qwen3-4B-Base"
DEVICE = "cuda"
BS = 16                         # batch size; lower to 8 if you get out-of-memory errors
TRAIN_VARIANTS = range(0, 16)   # used only to learn directions
EVAL_VARIANTS = range(16, 20)   # held out; used for every score. Raise to range(16, 24) for the final run
LAYER_STEP = 3                  # patching is cheap enough for every 3rd layer
ALPHAS = [1.0, 2.0, 4.0]        # edit strength, in units of the (true - false) difference vector
POSITIONS = ["mediator", "after", "answer"]

for v in ["lm", "model", "W32"]:
    if v in globals():
        del globals()[v]
gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load(dtype):
    m = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map=DEVICE).eval()
    return m, m.get_output_embeddings().weight.float()     # float32 output layer

lm, W32 = load(torch.float16)
N_LAYERS = lm.config.num_hidden_layers
LAYERS = list(range(0, N_LAYERS, LAYER_STEP))
print("layers to sweep:", LAYERS)

YES = tok(" Yes", add_special_tokens=False).input_ids
NO = tok(" No", add_special_tokens=False).input_ids
assert len(YES) == 1 and len(NO) == 1
YES, NO = YES[0], NO[0]

In [ ]:
# ---------- Prompt building, token positions, belief with optional edit
import ast
FEWSHOT = (
    "Story: The kettle was on the stove and the water started to bubble.\n"
    "Question: Was the water hot?\nAnswer: Yes\n\n"
    "Story: Maya left her umbrella at home and the sky stayed clear all day.\n"
    "Question: Did Maya get wet from rain?\nAnswer: No\n\n"
)
STORY_OFFSET = len(FEWSHOT) + len("Story: ")

def prompt(story, q):
    return f"{FEWSHOT}Story: {story}\nQuestion: {q}\nAnswer:"

def encode(stories, qs, spans):
    """Tokenise a batch (left-padded) and find, per row, the last token of the key sentence."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True,
              return_offsets_mapping=True)
    offs = enc.pop("offset_mapping")
    am = enc["attention_mask"]
    key_idx = []
    for i, (a, b) in enumerate(spans):
        a, b = a + STORY_OFFSET, b + STORY_OFFSET
        hit = ((offs[i, :, 0] < b) & (offs[i, :, 1] > a) & (am[i] == 1)).nonzero().flatten()
        assert len(hit) > 0, "key sentence not found in tokens"
        key_idx.append(hit[-1].item())
    return enc, torch.tensor(key_idx)

def position_mask(key_idx, T, mode):
    B = len(key_idx)
    m = torch.zeros(B, T)
    for i, k in enumerate(key_idx.tolist()):
        if mode == "mediator":
            m[i, k] = 1
        elif mode == "after":
            m[i, k + 1:T - 1] = 1     # tokens after the key sentence, excluding the final token
        elif mode == "answer":
            m[i, T - 1] = 1
    return m

def _layer_out(out):
    return out[0] if isinstance(out, tuple) else out

def belief(stories, qs, spans, edit=None, bs=BS):
    """log P(Yes) - log P(No). edit = dict(layer=int, vecs=[N, H] tensor, mode=str) or None."""
    res = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        handle = None
        if edit is not None:
            T = enc["input_ids"].shape[1]
            mask = position_mask(key_idx, T, edit["mode"]).to(DEVICE, lm.dtype)
            vecs = edit["vecs"][i:i+bs].to(DEVICE, lm.dtype)
            def hook(mod, inp, out, mask=mask, vecs=vecs):
                h = _layer_out(out) + mask[:, :, None] * vecs[:, None, :]
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            handle = lm.model.layers[edit["layer"]].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            if handle is not None:
                handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        res += (logp[:, YES] - logp[:, NO]).tolist()
    return np.array(res)

def key_states(stories, qs, spans, layers, bs=BS):
    """Residual stream (output of each layer in `layers`) at the last token of the key sentence."""
    out = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        rows = torch.arange(len(key_idx))
        out.append(torch.stack([hs[L + 1][rows, key_idx.to(DEVICE)].float().cpu() for L in layers], 1))
    return torch.cat(out)        # [N, len(layers), H]

# float16 overflow check; fall back to bfloat16 if needed
t = belief(["Priya ate a huge lunch and felt full."], ["Was Priya full?"], [[0, 5]])
if not np.isfinite(t).all():
    print("float16 overflow -> bfloat16")
    del lm, W32; gc.collect(); torch.cuda.empty_cache()
    lm, W32 = load(torch.bfloat16)
print("dtype:", lm.dtype)

d = pd.read_csv("stories_chain.csv")  # (not used below; pairs_chain.csv is loaded next)
d["variant"] = d.item_id.str.rsplit("_", n=1).str[1].astype(int)
for c in [c for c in d.columns if c.startswith("span_")]:
    d[c] = d[c].apply(ast.literal_eval)
train = d[d.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
ev = d[d.variant.isin(EVAL_VARIANTS)].reset_index(drop=True)
SCEN = sorted(d.scenario_id.unique())
print(f"{len(SCEN)} scenarios | train items {len(train)} | eval items {len(ev)}")

In [ ]:
# ---------- Settings
SET = "exploration"            # "exploration" first, then "confirmation"
EVAL = range(16, 20) if SET == "exploration" else range(16, 24)
MODES = ["word", "sentence"]
KINDS = ["true", "false", "unrelated"]
FILLERS = ["Was there a cat nearby?", "Did someone in the story have a brother?", "Was a radio playing?"]
TAG = f"P_{SET}"

import glob
from scipy.stats import wilcoxon
P = pd.read_csv("pairs_chain.csv")
for c in [c for c in P.columns if c.startswith(("span_", "wspan_"))]:
    P[c] = P[c].apply(ast.literal_eval)
P = P[(P.set == SET) & P.variant.isin(EVAL)].reset_index(drop=True)
for i, f in enumerate(FILLERS):
    P[f"q_f{i}"] = f
PROBES = {"cause": "q_cause", "effect": "q_eff", "med": "q_med", **{f"f{i}": f"q_f{i}" for i in range(len(FILLERS))}}
print(f"{SET}: {P.scenario_id.nunique()} scenarios, {len(P)} items | layers {LAYERS}")

## Step 1: the model's own seeing and doing, from the neutral base story

In [ ]:
path = f"{OUT}/{TAG}_reference_items.csv"
if os.path.exists(path):
    ref = pd.read_csv(path)
else:
    ref = P[["scenario_id", "item_id"]].copy()
    for cond in ["base", "see", "do", "false"]:
        for p, q in PROBES.items():
            ref[f"{p}_{cond}"] = belief(P[cond].tolist(), P[q].tolist(), P[f"span_{cond}"].tolist())
    ref.to_csv(path, index=False)
nf = len(FILLERS)
for cond in ["see", "do", "false"]:
    fill = sum(ref[f"f{i}_{cond}"] - ref[f"f{i}_base"] for i in range(nf)) / nf
    ref[f"dfill_{cond}"] = fill
    for p in ["cause", "effect", "med"]:
        ref[f"d{p}C_{cond}"] = ref[f"{p}_{cond}"] - ref[f"{p}_base"] - fill
sc = ref.groupby("scenario_id")[[c for c in ref.columns if c.startswith("d")]].mean()
sc["r_see"] = sc.dcauseC_see / sc.deffectC_see
sc["r_do"] = sc.dcauseC_do / sc.deffectC_do
sc["valid"] = ((sc.dcauseC_see > 0.5) & (sc.deffectC_see > 0.5) & (sc.deffectC_do > 0.5) & (sc.r_see - sc.r_do > 0.25))
sc.to_csv(f"{OUT}/{TAG}_reference_scenarios.csv")
display(sc[["dmedC_see", "dcauseC_see", "deffectC_see", "dcauseC_do", "deffectC_do", "dmedC_false", "dfill_see", "r_see", "r_do", "valid"]].round(2))
VALID = sc.index[sc.valid].tolist()
print(f"valid scenarios: {len(VALID)} / {len(sc)}")

## Step 2: record the source states, then patch
Source states are recorded once per item at every layer (they do not depend on the question, because the model reads left to right).

In [ ]:
def key_positions(stories, qs, spans, wspans):
    """Per row: index of the last token of the key word, and of the sentence end."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True, return_offsets_mapping=True)
    offs = enc.pop("offset_mapping"); am = enc["attention_mask"]
    wl, se = [], []
    for i, ((a, b), (wa, wb)) in enumerate(zip(spans, wspans)):
        def last_hit(x, y):
            x, y = x + STORY_OFFSET, y + STORY_OFFSET
            h = ((offs[i, :, 0] < y) & (offs[i, :, 1] > x) & (am[i] == 1)).nonzero().flatten()
            assert len(h) > 0
            return h[-1].item()
        wl.append(last_hit(wa, wb)); se.append(last_hit(a, b))
    return enc, wl, se

def record(stories, qs, spans, wspans, bs=BS):
    """States at [key-word-last ... sentence-end] for every layer in LAYERS. Returns list of [L, n_pos, H] tensors."""
    out = []
    for i in range(0, len(stories), bs):
        enc, wl, se = key_positions(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs], wspans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        for r in range(len(wl)):
            out.append(torch.stack([hs[L + 1][r, wl[r]:se[r] + 1].float().cpu() for L in LAYERS]))
    return out

def belief_patch(stories, qs, spans, wspans, src, li, mode, bs=BS):
    """Run the target stories with states replaced at layer LAYERS[li]. src[row] = [L, n_pos, H]."""
    res, skipped = [], 0
    L = LAYERS[li]
    for i in range(0, len(stories), bs):
        enc, wl, se = key_positions(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs], wspans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        items = []
        for r in range(len(wl)):
            s_states = src[i + r][li]                         # [n_pos_src, H]
            n_tgt = se[r] - wl[r] + 1
            if mode == "word":
                items.append(([wl[r]], s_states[:1]))
            elif s_states.shape[0] == n_tgt:
                items.append((list(range(wl[r], se[r] + 1)), s_states))
            else:
                items.append(None); skipped += 1
        def hook(mod, inp, out, items=items):
            h = _layer_out(out).clone()
            for r, it in enumerate(items):
                if it is not None:
                    h[r, it[0]] = it[1].to(h.device, h.dtype)
            return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
        handle = lm.model.layers[L].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        b = (logp[:, YES] - logp[:, NO]).tolist()
        res += [np.nan if it is None else v for it, v in zip(items, b)]
    return np.array(res), skipped

EVP = P[P.scenario_id.isin(VALID)].reset_index(drop=True)
src_rows = {"true": EVP.index.tolist(), "false": EVP.index.tolist()}
# unrelated source: the next valid scenario's true story with the same variant number
other = {s: VALID[(VALID.index(s) + 1) % len(VALID)] for s in VALID}
lookup = {(r.scenario_id, r.variant): i for i, r in P.iterrows()}
unrel_idx = [lookup[(other[r.scenario_id], r.variant)] for _, r in EVP.iterrows()]

t0 = time.time()
q0 = EVP.q_med.tolist()
SRC = {"true": record(EVP.see.tolist(), q0, EVP.span_see.tolist(), EVP.wspan_see.tolist()),
       "false": record(EVP.false.tolist(), q0, EVP.span_false.tolist(), EVP.wspan_false.tolist())}
U = P.loc[unrel_idx]
SRC["unrelated"] = record(U.see.tolist(), U.q_med.tolist(), U.span_see.tolist(), U.wspan_see.tolist())
print(f"recorded source states for {len(EVP)} items ({(time.time() - t0) / 60:.1f} min)")

stories, spans, wspans = EVP.base.tolist(), EVP.span_base.tolist(), EVP.wspan_base.tolist()
base_b = {p: belief(stories, EVP[q].tolist(), spans) for p, q in PROBES.items()}
print(f"{len(EVP)} items x {len(LAYERS) * len(MODES) * len(KINDS) * len(PROBES)} patched runs")
for li, L in enumerate(LAYERS):
    path = f"{OUT}/{TAG}_sweep_layer{L:02d}.csv"
    if os.path.exists(path):
        print("skip layer", L); continue
    rows = []
    for mode in MODES:
        for kind in KINDS:
            r = EVP[["scenario_id", "item_id"]].copy()
            r["layer"], r["mode"], r["kind"] = L, mode, kind
            for p, q in PROBES.items():
                b, skipped = belief_patch(stories, EVP[q].tolist(), spans, wspans, SRC[kind], li, mode)
                r[f"d{p}"] = b - base_b[p]
            r["skipped"] = skipped
            rows.append(r)
    pd.concat(rows).to_csv(path, index=False)
    print(f"layer {L} done ({(time.time() - t0) / 60:.1f} min)")

## Step 3: results
1. **Layer profile** of the true patch: mediator, cause and effect shifts (filler-corrected) by layer, with the unrelated patch for comparison.
2. **Pass table** using the fixed rule.
3. **D** for every setting (trust passing settings with n ≥ 8), with bootstrap CIs over scenarios.

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
def boot_median(x, n=2000):
    x = pd.Series(x).dropna().values
    if len(x) < 3:
        return np.nan, np.nan
    m = [np.median(rng.choice(x, len(x))) for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

sw = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{OUT}/{TAG}_sweep_layer*.csv")))
sw["dfill"] = sw[[f"df{i}" for i in range(len(FILLERS))]].mean(1)
for p in ["cause", "effect", "med"]:
    sw[f"d{p}C"] = sw[f"d{p}"] - sw.dfill
g = sw.groupby(["scenario_id", "layer", "mode", "kind"])[["dcauseC", "deffectC", "dmedC", "dfill"]].mean().reset_index()
g.to_csv(f"{OUT}/{TAG}_scenario_means.csv", index=False)
print("items skipped in sentence mode (length mismatch):", int(sw[sw["mode"] == "sentence"].groupby(["layer", "kind"]).skipped.max().max()))

# 1. layer profile
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, mode in zip(axes, MODES):
    m = g[g["mode"] == mode].groupby(["kind", "layer"])[["dmedC", "dcauseC", "deffectC", "dfill"]].mean()
    for col, ls in [("dmedC", "-"), ("dcauseC", "--"), ("deffectC", ":")]:
        ax.plot(m.loc["true"].index, m.loc["true"][col], ls, color="C0", label=f"true patch: {col[1:-1]}")
    ax.plot(m.loc["unrelated"].index, m.loc["unrelated"]["dmedC"], "-", color="C7", label="unrelated patch: med")
    ax.plot(m.loc["false"].index, m.loc["false"]["dmedC"], "-", color="C3", label="false patch: med")
    ax.plot(m.loc["true"].index, m.loc["true"]["dfill"], "-", color="C2", alpha=.5, label="true patch: filler")
    ax.axhline(0, color="k", lw=.5); ax.set_title(f"{mode} patch"); ax.set_xlabel("layer")
axes[0].set_ylabel("shift in log-odds (filler-corrected)"); axes[1].legend(fontsize=7)
plt.suptitle(f"Activation patching, {SET} set"); plt.tight_layout()
plt.savefig(f"{OUT}/{TAG}_layer_profile.png", dpi=150, bbox_inches="tight"); plt.show()

# 2. pass table
rows = []
for (L, mode), grp in g.groupby(["layer", "mode"]):
    t = grp[grp.kind == "true"].set_index("scenario_id"); u = grp[grp.kind == "unrelated"].set_index("scenario_id")
    f = grp[grp.kind == "false"].set_index("scenario_id")
    diff = (t.dmedC - u.dmedC).dropna()
    p = wilcoxon(diff).pvalue if (diff != 0).sum() > 5 else np.nan
    rows.append(dict(layer=L, mode=mode, dmedC_true=t.dmedC.mean(), dmedC_unrel=u.dmedC.mean(), dmedC_false=f.dmedC.mean(),
                     true_minus_unrel=diff.mean(), share_true_wins=(diff > 0).mean(), wilcoxon_p=p, dfill_true=t.dfill.mean(),
                     dcauseC_true=t.dcauseC.mean(), deffectC_true=t.deffectC.mean(),
                     passes=bool((t.dmedC.mean() >= 1.0) and (diff.mean() > 0.5) and ((diff > 0).mean() >= 0.7)
                                 and (abs(t.dfill.mean()) < 0.5) and (f.dmedC.mean() < 0))))
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/{TAG}_settings.csv", index=False)
display(res.round(2))
print(f"{int(res.passes.sum())} of {len(res)} settings pass:", res[res.passes][["layer", "mode"]].values.tolist())

# 3. D
t = g[g.kind == "true"].merge(sc[["deffectC_see", "r_see", "r_do"]], left_on="scenario_id", right_index=True)
t["works"] = t.deffectC >= 0.5 * t.deffectC_see
t["r_edit"] = t.dcauseC / t.deffectC
t["D"] = np.where(t.works, (t.r_see - t.r_edit) / (t.r_see - t.r_do), np.nan)
t["D_clip"] = t.D.clip(0, 1)
t.to_csv(f"{OUT}/{TAG}_scenario_scores.csv", index=False)
out = []
for (L, mode), grp in t.groupby(["layer", "mode"]):
    lo, hi = boot_median(grp.D)
    out.append(dict(layer=L, mode=mode, works_rate=grp.works.mean(), n_scored=int(grp.D.notna().sum()),
                    D_median=grp.D.median(), D_clip_median=grp.D_clip.median(), ci_lo=lo, ci_hi=hi))
dres = pd.DataFrame(out).merge(res[["layer", "mode", "passes"]], on=["layer", "mode"])
dres.to_csv(f"{OUT}/{TAG}_D_summary.csv", index=False)
display(dres.round(2))

fig, ax = plt.subplots(figsize=(6, 4))
for mode, c in zip(MODES, ["C0", "C1"]):
    x = dres[dres["mode"] == mode]
    ok = x[x.n_scored >= 5]
    ax.errorbar(ok.layer, ok.D_median, yerr=[ok.D_median - ok.ci_lo, ok.ci_hi - ok.D_median], fmt="o-", color=c, capsize=3, label=f"{mode} patch")
ax.axhline(0, color="k", lw=.5, ls="--"); ax.axhline(1, color="k", lw=.5, ls="--")
ax.text(ax.get_xlim()[1], 0, " seeing", va="center"); ax.text(ax.get_xlim()[1], 1, " doing", va="center")
ax.set_xlabel("layer"); ax.set_ylabel("median D (95% CI)"); ax.set_title(f"Do-likeness of patches, {SET} set (n >= 5 scenarios)"); ax.legend()
plt.savefig(f"{OUT}/{TAG}_D_by_layer.png", dpi=150, bbox_inches="tight"); plt.show()